# RK1828 Depth Anything V3 多视角深度估计 Demo


[Depth Anything V3](https://github.com/ByteDance-Seed/depth-anything-3)（字节跳动）
多视角深度估计。模型被拆成三个子模型（local / global / head），固定输入 **10 个视角**，
输出每个视角的深度热力图。RK1828 上 10 视角合计约 12 秒。

| 步骤 | 内容 |
|---|---|
| 1 | 下载模型和 10 张示例图（可选 HF / HF-Mirror 源） |
| 2 | 解压预编译 demo |
| 3 | 准备图片列表并运行 |
| 4 | 查看深度图 |

## 0. 环境检查

先确认 RKNN3 运行库（`librknn3_api.so`）和 RK1828 设备都正常。
如失败，请先安装驱动/运行时（`rknn3-rk182x` DEB 包），并用 `sudo rknn-smi info` 确认能看到 RK1828。

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # 加载 /usr/lib/librknn3_api.so；失败说明 Runtime 未安装

devices = find_devices()
if not devices:
    raise RuntimeError("未发现 RK1828 设备！请检查: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"发现设备: id={dev_id}  接口={dev_type}")

## 1. 下载模型

本 demo 的模型文件托管在 Hugging Face（`qhddz` 账号）。
**先用下拉框选择下载源**，再运行下一个单元格开始下载：

- **HF-Mirror 国内镜像**：`hf-mirror.com`，国内直连，推荐
- **Hugging Face 官方**：`huggingface.co`，海外或已有代理时用

> 已存在（且非空）的文件会**自动跳过**，重复运行不会重新下载。

In [ ]:
from nb_download import source_picker

# (HF仓库名, 仓库内路径, 本地路径)
FILES = [
    ("depth-anything-v3-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/depth_anything_v3/demo_x86_64.tar.gz"),
    ("depth-anything-v3-rknn3-rk1828", "model/da3_base_global.rknn", "demos/depth_anything_v3/model/da3_base_global.rknn"),
    ("depth-anything-v3-rknn3-rk1828", "model/da3_base_global.weight", "demos/depth_anything_v3/model/da3_base_global.weight"),
    ("depth-anything-v3-rknn3-rk1828", "model/da3_base_head.rknn", "demos/depth_anything_v3/model/da3_base_head.rknn"),
    ("depth-anything-v3-rknn3-rk1828", "model/da3_base_head.weight", "demos/depth_anything_v3/model/da3_base_head.weight"),
    ("depth-anything-v3-rknn3-rk1828", "model/da3_base_local.rknn", "demos/depth_anything_v3/model/da3_base_local.rknn"),
    ("depth-anything-v3-rknn3-rk1828", "model/da3_base_local.weight", "demos/depth_anything_v3/model/da3_base_local.weight"),
    ("depth-anything-v3-rknn3-rk1828", "model/frame_01.jpg", "demos/depth_anything_v3/model/frame_01.jpg"),
    ("depth-anything-v3-rknn3-rk1828", "model/frame_02.jpg", "demos/depth_anything_v3/model/frame_02.jpg"),
    ("depth-anything-v3-rknn3-rk1828", "model/frame_03.jpg", "demos/depth_anything_v3/model/frame_03.jpg"),
    ("depth-anything-v3-rknn3-rk1828", "model/frame_04.jpg", "demos/depth_anything_v3/model/frame_04.jpg"),
    ("depth-anything-v3-rknn3-rk1828", "model/frame_05.jpg", "demos/depth_anything_v3/model/frame_05.jpg"),
    ("depth-anything-v3-rknn3-rk1828", "model/frame_06.jpg", "demos/depth_anything_v3/model/frame_06.jpg"),
    ("depth-anything-v3-rknn3-rk1828", "model/frame_07.jpg", "demos/depth_anything_v3/model/frame_07.jpg"),
    ("depth-anything-v3-rknn3-rk1828", "model/frame_08.jpg", "demos/depth_anything_v3/model/frame_08.jpg"),
    ("depth-anything-v3-rknn3-rk1828", "model/frame_09.jpg", "demos/depth_anything_v3/model/frame_09.jpg"),
    ("depth-anything-v3-rknn3-rk1828", "model/frame_10.jpg", "demos/depth_anything_v3/model/frame_10.jpg"),
    ("depth-anything-v3-rknn3-rk1828", "model/images.txt", "demos/depth_anything_v3/model/images.txt"),
]

w_source = source_picker()

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, w_source.value)

## 2. 解压预编译 demo

`demo_x86_64.tar.gz` 里是官方流程编译好的 x86_64 C++ demo 和运行时库（`lib/`）。
HF 不允许直接托管可执行文件，所以打包成 tar.gz，解压一次即可（重复运行自动跳过）。

> **为什么这个 demo 用 C++ 而不是纯 Python？**
> 01-05（分类/检测/FastSAM/pose/分割）是单个模型一次前向，后处理简单，
> 所以用 Python（`rknn3.py`）逐步演示。而本 demo 的流水线复杂得多——
> 多模型串联、tokenizer、KV cache、滑窗/流式状态等，前后处理和编排逻辑
> 有几百上千行，重写 Python 版成本高且容易和官方实现不一致。
> 因此直接运行官方 `rknn3-model-zoo` 编译的 C++ demo，
> 源码见 [github.com/airockchip/rknn3-model-zoo](https://github.com/airockchip/rknn3-model-zoo)
> 的 `examples/` 目录。

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/depth_anything_v3/demo_x86_64.tar.gz", "demos/depth_anything_v3")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. 准备 10 视角图片列表并运行

仓库自带同一场景的 10 张示例图（`model/frame_01.jpg` ~ `frame_10.jpg`），
demo 需要一个恰好 10 行的图片路径列表文件。

In [ ]:
import glob, os

DEMO = "demos/depth_anything_v3"
frames = sorted(glob.glob(os.path.join(DEMO, "model/frame_*.jpg")))
assert len(frames) == 10, f"需要恰好 10 张图，当前 {len(frames)} 张"
with open(os.path.join(DEMO, "image_list.txt"), "w") as f:
    f.write("\n".join(os.path.abspath(p) for p in frames))  # demo 要求绝对路径
show_images(frames[:5], max_width=320)
print(f"共 {len(frames)} 张，列表已写入 image_list.txt")

In [ ]:
run_demo(["./rknn_da3_base_demo",
          "model/da3_base_local.rknn", "model/da3_base_global.rknn",
          "model/da3_base_head.rknn",
          "image_list.txt", "10", "depth_vis"], DEMO)

## 4. 查看深度热力图（越亮越近）

In [ ]:
import glob

depth_imgs = sorted(glob.glob(os.path.join(DEMO, "depth_vis/*")))
print(f"生成 {len(depth_imgs)} 张深度图")
show_images(depth_imgs[:5], max_width=320)

## 小结

- 大模型可以拆成多个子模型分别转换、流水线运行，这是 RKNN 部署超大模型的常见手段。
- 换自己的照片时注意：必须恰好 10 个视角，且尽量是同一场景不同角度。